# Statistics for Model Evaluation: Confidence Intervals and Significance Testing

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset), loaded via `sklearn.datasets.load_breast_cancer` — 569 real patient samples, 30 features computed from digitized images of a fine needle aspirate (FNA) of a breast mass, label = malignant/benign. This is a real diagnostic dataset with a real (and clinically meaningful) class imbalance, not a synthetic toy.

**Problem statement.** In ML quality engineering, a single accuracy number is never enough evidence to ship or reject a model — an observed difference in performance might be real, or it might just be noise from one particular train/test split. In this exercise you will train two classifiers on the Breast Cancer Wisconsin dataset, then apply core statistical tools — bootstrap confidence intervals, a paired significance test, and a calibration check — to decide, with a stated confidence level, whether one model is genuinely better than the other, and where each model's evaluation metrics could mislead a reviewer who only looked at the headline number.

**What you should produce (be specific):**
1. **Bootstrap 95% confidence intervals** (≥ 2000 resamples) for both **accuracy** and **ROC-AUC**, for each of the two models, on the held-out test set.
2. A **paired McNemar's test** comparing the two models' predictions on the *same* test set — state the null hypothesis, compute the test statistic and p-value, and give a conclusion at α = 0.05.
3. A short written report (3-5 sentences, in a markdown cell) stating: which model you would ship, whether the difference in performance is statistically significant, and **one concrete metric failure mode** you found in this analysis (e.g., a CI wide enough to make the "better" model's ranking unreliable, or a metric that hides poor performance on the minority class).
4. A **calibration check** (reliability curve and/or Brier score) for the better model's predicted probabilities, with one sentence on whether those probabilities can be trusted at face value (relevant if this model's output were used as a risk score rather than just a class label).


## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss

RNG = np.random.default_rng(42)


## Load the data and hold out a test set

We stratify the split so both train and test sets preserve the real class balance (the dataset is skewed toward the "benign" class, which matters later).

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

print(X.shape, y.value_counts(normalize=True).to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


## Train two candidate models

A simple linear model and a tree ensemble — fitted for you, since the point of this exercise is the *statistics*, not model selection.

In [ ]:
models = {
    "logreg": LogisticRegression(max_iter=5000, random_state=42).fit(X_train, y_train),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train),
}

preds = {name: m.predict(X_test) for name, m in models.items()}
probas = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}

for name in models:
    print(name, "accuracy:", accuracy_score(y_test, preds[name]),
          "roc_auc:", roc_auc_score(y_test, probas[name]))


## Your turn: fill in the TODOs below

Implement the three statistical tools described in the problem statement. Don't peek at the solution until you've tried each one.

In [ ]:
def bootstrap_ci(y_true, y_pred, y_proba, metric_fn, n_boot=2000, alpha=0.05, uses_proba=False, random_state=42):
    """
    TODO: Compute a bootstrap confidence interval for `metric_fn` evaluated on
    (y_true, y_pred) or (y_true, y_proba) if `uses_proba` is True.

    Steps:
      1. Resample the *indices* of the test set with replacement, n_boot times.
      2. For each resample, compute metric_fn on the resampled arrays.
      3. Return (point_estimate, lower, upper) using the alpha/2 and 1-alpha/2
         percentiles of the bootstrap distribution.

    Handle the edge case where a resample contains only one class (roc_auc_score
    will raise in that case) -- skip that resample rather than crashing.
    """
    raise NotImplementedError


In [ ]:
def mcnemars_test(y_true, pred_a, pred_b):
    """
    TODO: Run McNemar's test comparing two models' predictions on the same
    test set.

    Steps:
      1. Build the 2x2 table of (model_a correct/incorrect) x (model_b correct/incorrect).
      2. Use the discordant cells (a correct & b wrong) vs (a wrong & b correct).
      3. Compute the test statistic and p-value (use the exact binomial version
         when the number of discordant pairs is small, e.g. < 25; otherwise the
         chi-square approximation with continuity correction is fine).
      4. Return (statistic, p_value, discordant_counts).

    State your null hypothesis in a comment before you start coding.
    """
    raise NotImplementedError


In [ ]:
# TODO: call bootstrap_ci for accuracy and roc_auc, for both models.
# Print each as: "<model> <metric>: point=.., 95% CI=[.., ..]"


# TODO: call mcnemars_test comparing logreg vs rf predictions on y_test.
# Print the null hypothesis, statistic, p-value, and your conclusion at alpha=0.05.


In [ ]:
# TODO: calibration check for the better model (per your bootstrap/McNemar results).
# Compute the Brier score, and/or bin predicted probabilities into ~10 buckets
# and compare each bucket's mean predicted probability to its observed positive rate
# (a manual reliability curve, or use sklearn.calibration.calibration_curve if available).


*(Write your 3-5 sentence report from deliverable #3 here, once you have your results.)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
from scipy import stats

def bootstrap_ci(y_true, y_pred, y_proba, metric_fn, n_boot=2000, alpha=0.05, uses_proba=False, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba)
    n = len(y_true)

    point = metric_fn(y_true, y_proba if uses_proba else y_pred)

    boot_scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue  # can't score AUC (or a stable accuracy) on a single-class resample
        arr = y_proba[idx] if uses_proba else y_pred[idx]
        boot_scores.append(metric_fn(yt, arr))

    lower, upper = np.percentile(boot_scores, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper, len(boot_scores)


def mcnemars_test(y_true, pred_a, pred_b):
    # H0: the two models have the same error rate (disagreements are 50/50 in either direction).
    y_true = np.asarray(y_true)
    a_correct = (pred_a == y_true)
    b_correct = (pred_b == y_true)

    b01 = int(np.sum(a_correct & ~b_correct))  # a right, b wrong
    b10 = int(np.sum(~a_correct & b_correct))  # a wrong, b right
    n_discordant = b01 + b10

    if n_discordant == 0:
        return 0.0, 1.0, (b01, b10)

    if n_discordant < 25:
        # exact binomial test on the discordant pairs
        p_value = 2 * stats.binom.cdf(min(b01, b10), n_discordant, 0.5)
        p_value = min(p_value, 1.0)
        statistic = min(b01, b10)
    else:
        statistic = (abs(b01 - b10) - 1) ** 2 / n_discordant  # continuity-corrected chi-square
        p_value = 1 - stats.chi2.cdf(statistic, df=1)

    return statistic, p_value, (b01, b10)


# --- Bootstrap CIs ---
for name in models:
    acc_point, acc_lo, acc_hi, n_used = bootstrap_ci(y_test, preds[name], probas[name], accuracy_score)
    auc_point, auc_lo, auc_hi, _ = bootstrap_ci(y_test, preds[name], probas[name], roc_auc_score, uses_proba=True)
    print(f"{name} accuracy: point={acc_point:.3f}, 95% CI=[{acc_lo:.3f}, {acc_hi:.3f}]")
    print(f"{name} roc_auc : point={auc_point:.3f}, 95% CI=[{auc_lo:.3f}, {auc_hi:.3f}]")

# --- McNemar's test ---
stat, p_value, (b01, b10) = mcnemars_test(y_test, preds["logreg"], preds["rf"])
print(f"McNemar statistic={stat:.3f}, p={p_value:.4f}, discordant=(logreg-only-right={b01}, rf-only-right={b10})")
print("Reject H0 (models differ) at alpha=0.05" if p_value < 0.05 else "Fail to reject H0: no significant difference detected")

# --- Calibration check (Brier score) for the better model ---
better = "logreg" if roc_auc_score(y_test, probas["logreg"]) >= roc_auc_score(y_test, probas["rf"]) else "rf"
brier = brier_score_loss(y_test, probas[better])
print(f"Brier score for {better}: {brier:.4f} (lower is better; 0 = perfect, 0.25 = no-skill baseline for balanced classes)")
```

**Explanation.** The bootstrap CIs are usually wide relative to the gap between the two models' point estimates — with only ~170 test samples, accuracy and even ROC-AUC differences of a few points are frequently *not* statistically distinguishable, which McNemar's test (run on the same test set, so it isolates genuine disagreement rather than sampling noise between two separate test sets) will often confirm with p > 0.05. That's the metric failure mode this exercise is built to surface: a leaderboard ranking built on point estimates alone can flip if you resample the test set, and a "winner" chosen without a significance test may just be picking up split-specific noise. The Brier score check matters separately from accuracy/AUC: a model can rank cases correctly (good AUC) while still being poorly calibrated (e.g., consistently over- or under-confident), which is exactly the kind of thing accuracy alone will never reveal but matters a great deal if the probability itself is used downstream (e.g., as a triage risk score).

</details>